# MD-GRTN (No Diffusion, Single Sequence) — Paper-Aligned Implementation

- MAF → MGRC → STFormer → Output
- Mixed precision (T4-ready)
- Dynamic + Distance Graph

Updated: 2026-03-25


In [ ]:

import torch, numpy as np, random
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


In [ ]:

data = np.load('/content/your_dataset.npz')
X, Y = data['x'], data['y']
coords = data['coords'] if 'coords' in data.files else None


In [ ]:

class DatasetWrap(Dataset):
    def __init__(self,X,Y):
        self.X = torch.tensor(X,dtype=torch.float32)
        self.Y = torch.tensor(Y,dtype=torch.float32)
    def __len__(self): return len(self.X)
    def __getitem__(self,i): return self.X[i], self.Y[i]


In [ ]:

class MAF(nn.Module):
    def __init__(self,dim):
        super().__init__()
        self.q=nn.Linear(dim,dim)
        self.k=nn.Linear(dim,dim)
        self.v=nn.Linear(dim,dim)
    def forward(self,x):
        Q,K,V=self.q(x),self.k(x),self.v(x)
        attn=torch.softmax(Q@K.transpose(-1,-2)/x.shape[-1]**0.5,-1)
        return attn@V


In [ ]:

class GraphConstructor(nn.Module):
    def __init__(self,N):
        super().__init__()
        self.E1=nn.Parameter(torch.randn(N,16))
        self.E2=nn.Parameter(torch.randn(N,16))
    def forward(self):
        A=torch.relu(self.E1@self.E2.T)
        return torch.softmax(A,-1)


In [ ]:

class MGRC(nn.Module):
    def __init__(self,dim):
        super().__init__()
        self.lin=nn.Linear(dim,dim)
        self.gru=nn.GRU(dim,dim,batch_first=True)
    def forward(self,x,A):
        B,T,N,F=x.shape
        out=[]
        for t in range(T):
            xt=self.lin(A@x[:,t])
            out.append(xt)
        x=torch.stack(out,1)
        x=x.view(B*N,T,F)
        x,_=self.gru(x)
        return x.view(B,N,T,F).permute(0,2,1,3)


In [ ]:

class STFormer(nn.Module):
    def __init__(self,dim,N,T):
        super().__init__()
        self.sa=nn.MultiheadAttention(dim,4,batch_first=True)
        self.ta=nn.MultiheadAttention(dim,4,batch_first=True)
        self.se=nn.Parameter(torch.randn(N,dim))
        self.te=nn.Parameter(torch.randn(T,dim))
    def forward(self,x,A):
        B,T,N,F=x.shape
        x=x+torch.matmul(A,self.se)
        x=x.permute(0,2,1,3).reshape(B*N,T,F)
        x,_=self.sa(x,x,x)
        x=x+self.te[:T]
        x,_=self.ta(x,x,x)
        return x.view(B,N,T,F).permute(0,2,1,3)


In [ ]:

class Model(nn.Module):
    def __init__(self,N,F,T):
        super().__init__()
        self.inp=nn.Linear(F,64)
        self.maf=MAF(64)
        self.gc=GraphConstructor(N)
        self.mgrc=MGRC(64)
        self.st=STFormer(64,N,T)
        self.fc=nn.Linear(64,1)
    def forward(self,x):
        x=self.inp(x)
        x=self.maf(x)
        A=self.gc()
        x=self.mgrc(x,A)
        x=self.st(x,A)
        return self.fc(x)
